## Comparing dataset projections

In [120]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Dynamic World LULC

In [125]:
START = "2024-01-01"
END = "2024-12-31"

# 10m resolution

dw_col = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(panama_geom)
    .filterDate(START, END)
)

dw_mode = dw_col.select("label").mode()

VIS_PALETTE = [
    "419bdf", "397d49", "88b053", "7a87c6",
    "e49635", "dfc35a", "c4281b", "a59b8f", "b39fe1"
]

Map.addLayer(
    dw_mode.clip(panama_geom),
    {"min": 0, "max": 8, "palette": VIS_PALETTE},
    "Dynamic World"
)

Map.addLayer(
    dw_col.first().clip(panama_geom), 
    {"min": 0, "max": 8, "palette": VIS_PALETTE}, 
    "Dynamic World First Image"
) 

Map 

EEException: Image.visualize: Cannot provide a palette when visualizing more than one band.

In [116]:
dw_mode.projection().nominalScale().getInfo()

111319.49079327357

In [118]:
dw_col.first().projection().nominalScale().getInfo()

10

### Copernicus DEM (used in PRISM)

In [113]:
# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

# Create a single DEM image and clip it
elevation = (
    dataset
    .select('DEM')
    .first()
    .clip(panama_geom)
)

elevation_vis = {
    'min': 0,
    'max': 2500,
    'palette': ['0000ff', '00ffff', 'ffff00', 'ff0000', 'ffffff'],
}

# reprojecting to fit other data layers
sample_image = dw_mode
collection_projection = sample_image.projection()

reprojected_elevation = (
    elevation.resample("bilinear")
    .reproject(collection_projection)
    .rename("elevation_reprojected")
    .clip(panama_geom)
)

Map.addLayer(reprojected_elevation, elevation_vis, 'Panama DEM')

In [115]:
reprojected_elevation.projection().getInfo()

{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

In [ ]:
dataset.first().projection().getInfo()

30.922080775909325

In [95]:
elevation.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.0002777777777777778,
  0,
  -78.00013888888888,
  0,
  -0.0002777777777777778,
  8.00013888888889]}

### Distance to protected areas

In [96]:
pa_fc = ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(panama_geom)

pa_raster = ee.Image().byte().paint(pa_fc, 1)

dist_pa = (
    pa_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .clip(panama_geom)
)

Map.addLayer(pa_fc, {}, "Protected Areas")
Map.addLayer(dist_pa, {}, "Distance Protected Areas")

In [97]:
# pa_fc.projection().getInfo()
dist_pa.projection().getInfo()

{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

### Distance to roads

In [98]:
roads = ee.FeatureCollection("projects/deforestation-495419/assets/Panama_OSM_Roads")

roads_raster = ee.Image().byte().paint(roads, 1)

distance_roads = (
    roads_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .rename("dist_roads_m")
    .clip(panama_geom)
)

Map.addLayer(roads_raster, {"palette": ["black"]}, "Roads")
Map.addLayer(distance_roads, {"min": 0, "max": 5000}, "Distance to Roads")

In [99]:
distance_roads.projection().getInfo()

{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

### Distance to urban and rural areas

In [109]:
# 1000m resolution
smod = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2030").select("smod_code")

urban = smod.gte(21).And(smod.lte(30))
rural = smod.gte(11).And(smod.lte(13))

urban_img = urban.unmask(0).toByte()
rural_img = rural.unmask(0).toByte()

distance_urban = urban_img.distance(ee.Kernel.euclidean(50000, "meters")).clip(panama_geom)
distance_rural = rural_img.distance(ee.Kernel.euclidean(50000, "meters")).clip(panama_geom)


# reprojecting to fit other data layers
sample_image = dw_mode
collection_projection = sample_image.projection()

reprojected_urban = (
    distance_urban.resample("bilinear")
    .reproject(collection_projection)
    .rename("urban_reprojected")
    .clip(panama_geom)
)

reprojected_rural = (
    distance_rural.resample("bilinear")
    .reproject(collection_projection)
    .rename("rural_reprojected")
    .clip(panama_geom)
)

# Map.addLayer(distance_urban, {"min": 0, "max": 25000}, "Distance Urban")
# Map.addLayer(distance_rural, {"min": 0, "max": 25000}, "Distance Rural")
Map.addLayer(reprojected_urban, {"min": 0, "max": 25000}, "Urban Distance Reprojected")
Map.addLayer(reprojected_rural, {"min": 0, "max": 25000}, "Rural Distance Reprojected")

# Map.addLayer(urban.selfMask(), {"palette": ["red"]}, "Urban")
# Map.addLayer(rural.selfMask(), {"palette": ["blue"]}, "Rural")

In [111]:
# reprojected_rural.projection().getInfo()
reprojected_urban.projection().getInfo()


{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

### Forest type (dry vs wet)

In [104]:
# 10m resolution
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()

landcover = worldcover.select("Map")

forest = landcover.eq(10)

ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")

dry_ecoregion = ecoregions.filter(
    ee.Filter.eq("ECO_NAME", "Panamanian dry forests")
)

dry_mask = ee.Image.constant(0).paint(dry_ecoregion, 1).selfMask()

dry_forest = forest.updateMask(dry_mask)
wet_forest = forest.updateMask(dry_mask.unmask(0).Not())

classified = ee.Image(0)
classified = classified.where(wet_forest, 1)
classified = classified.where(dry_forest, 2)

Map.addLayer(
    classified.clip(panama_geom),
    {"min": 0, "max": 2, "palette": ["white", "006400", "d4a017"]},
    "Forest Type"
)

In [105]:
classified.projection().getInfo()

{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

### Generate Map

In [106]:
Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…